# Reorder Quantity Prediction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

In [ ]:
df = pd.read_csv('historical_data.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.describe().T

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['sold'] = pd.to_numeric(df['sold'], errors='coerce')
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['is_promotion'] = pd.to_numeric(df['is_promotion'], errors='coerce')
df['category'] = df['category'].fillna('Unknown').astype(str)
df['hcode'] = df['hcode'].fillna('Unknown').astype(str)
df = df.dropna(subset=['date', 'product_id', 'sold']).copy()
df = df.sort_values(['date', 'product_id']).reset_index(drop=True)

In [ ]:
print('Start date:', df['date'].min())
print('End date:', df['date'].max())
print('Products:', df['product_id'].nunique())
print('Categories:', df['category'].nunique())
print('H-Codes:', df['hcode'].nunique())

In [ ]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['day_of_week'] = df['date'].dt.dayofweek

In [ ]:
df['category'].value_counts().head(20)

## 2. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df['sold'], bins=50)
plt.title('Distribution of Units Sold')
plt.xlabel('Units Sold')
plt.ylabel('Frequency')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(np.log1p(df['sold'].clip(lower=0)), bins=50)
plt.title('Sales Distribution on Log Scale')
plt.xlabel('log(1 + Units Sold)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
df['sold'].describe()

In [ ]:
df['sold'].quantile([0.90, 0.95, 0.99, 0.995, 0.999])

In [ ]:
print('Rows above 100 units:', (df['sold'] > 100).sum())
print('Rows above 1,000 units:', (df['sold'] > 1000).sum())
df.sort_values('sold', ascending=False)[
    ['product_id', 'date', 'sold', 'price', 'is_promotion']
].head(20)

In [ ]:
product_eda = (
    df.groupby(['product_id', 'category'])['sold']
      .agg(
          records='size',
          total_sold='sum',
          median_sold='median',
          max_sold='max',
          mean_sold='mean'
      )
      .reset_index()
      .sort_values('max_sold', ascending=False)
)
product_eda.head(20)

In [ ]:
daily_sales = df.groupby('date')['sold'].sum()

plt.figure(figsize=(14, 5))
daily_sales.plot()
plt.title('Daily Sales Trend')
plt.xlabel('Date')
plt.ylabel('Units Sold')
plt.show()

In [ ]:
monthly_sales = df.groupby(df['date'].dt.to_period('M'))['sold'].sum()

plt.figure(figsize=(14, 5))
monthly_sales.plot()
plt.title('Monthly Sales Trend')
plt.xlabel('Month')
plt.ylabel('Units Sold')
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
df.boxplot(column='sold', by='is_promotion')
plt.title('Promotion vs Sales')
plt.suptitle('')
plt.xlabel('Promotion')
plt.ylabel('Units Sold')
plt.show()

In [ ]:
category_sales = df.groupby('category')['sold'].sum().sort_values(ascending=False)
category_sales.head(20).sort_values().plot(kind='barh', figsize=(10, 7))
plt.title('Top Categories by Total Sales')
plt.xlabel('Units Sold')
plt.ylabel('Category')
plt.show()

## 3. Daily Product Table

In [ ]:
full_dates = pd.date_range(
    start=df['date'].min(),
    end=df['date'].max(),
    freq='D'
)

all_products = df['product_id'].drop_duplicates().tolist()

idx = pd.MultiIndex.from_product(
    [all_products, full_dates],
    names=['product_id', 'date']
)

In [ ]:
daily_sales = (
    df.groupby(['product_id', 'date'])['sold']
      .sum()
      .reindex(idx, fill_value=0)
      .reset_index()
)

daily_sales.head()

In [ ]:
info = (
    df.sort_values(['product_id', 'date'])
      .groupby(['product_id', 'date'])[['price', 'is_promotion', 'hcode']]
      .last()
      .reset_index()
)

product_category = df[['product_id', 'category']].drop_duplicates('product_id')

In [ ]:
daily = daily_sales.merge(
    info,
    on=['product_id', 'date'],
    how='left'
)

daily = daily.merge(
    product_category,
    on='product_id',
    how='left'
)

daily = daily.sort_values(['product_id', 'date']).reset_index(drop=True)

In [ ]:
daily['price'] = daily.groupby('product_id')['price'].ffill()
daily['is_promotion'] = daily['is_promotion'].fillna(0)
daily['hcode'] = daily.groupby('product_id')['hcode'].ffill()
daily['hcode'] = daily['hcode'].fillna('Unknown').astype(str)
daily['category'] = daily['category'].fillna('Unknown').astype(str)
daily.head()

In [ ]:
print('Daily table shape:', daily.shape)
print('Missing categories:', daily['category'].isna().sum())
print('Categories:', daily['category'].nunique())

## 4. Outlier Analysis

In [ ]:
daily[['sold']].describe()

In [ ]:
daily['sold'].quantile([0.90, 0.95, 0.99, 0.995, 0.999])

In [ ]:
daily.sort_values('sold', ascending=False)[
    ['product_id', 'category', 'date', 'sold', 'price', 'is_promotion']
].head(20)

In [ ]:
product_p99 = (
    daily.groupby('product_id')['sold']
         .quantile(0.99)
         .rename('product_p99')
         .reset_index()
)

outlier_rows = daily.merge(product_p99, on='product_id', how='left')
outlier_rows = outlier_rows[outlier_rows['sold'] > outlier_rows['product_p99']]
outlier_rows.sort_values('sold', ascending=False).head(20)

## 5. Feature Engineering

In [ ]:
daily['lag_1'] = daily.groupby('product_id')['sold'].transform(lambda values: values.shift(1))
daily['lag_7'] = daily.groupby('product_id')['sold'].transform(lambda values: values.shift(7))
daily['lag_14'] = daily.groupby('product_id')['sold'].transform(lambda values: values.shift(14))

In [ ]:
daily['lag_28'] = daily.groupby('product_id')['sold'].transform(lambda values: values.shift(28))
daily['lag_56'] = daily.groupby('product_id')['sold'].transform(lambda values: values.shift(56))
daily['lag_365'] = daily.groupby('product_id')['sold'].transform(lambda values: values.shift(365))

In [ ]:
daily['mean_7'] = daily.groupby('product_id')['sold'].transform(
    lambda values: values.shift(1).rolling(7, min_periods=1).mean()
)
daily['mean_14'] = daily.groupby('product_id')['sold'].transform(
    lambda values: values.shift(1).rolling(14, min_periods=1).mean()
)

In [ ]:
daily['mean_30'] = daily.groupby('product_id')['sold'].transform(
    lambda values: values.shift(1).rolling(30, min_periods=1).mean()
)
daily['mean_60'] = daily.groupby('product_id')['sold'].transform(
    lambda values: values.shift(1).rolling(60, min_periods=1).mean()
)

In [ ]:
daily['median_7'] = daily.groupby('product_id')['sold'].transform(
    lambda values: values.shift(1).rolling(7, min_periods=1).median()
)
daily['median_30'] = daily.groupby('product_id')['sold'].transform(
    lambda values: values.shift(1).rolling(30, min_periods=1).median()
)
daily['median_60'] = daily.groupby('product_id')['sold'].transform(
    lambda values: values.shift(1).rolling(60, min_periods=1).median()
)

In [ ]:
daily['sum_7'] = daily.groupby('product_id')['sold'].transform(
    lambda values: values.shift(1).rolling(7, min_periods=1).sum()
)
daily['sum_14'] = daily.groupby('product_id')['sold'].transform(
    lambda values: values.shift(1).rolling(14, min_periods=1).sum()
)

In [ ]:
daily['sum_30'] = daily.groupby('product_id')['sold'].transform(
    lambda values: values.shift(1).rolling(30, min_periods=1).sum()
)
daily['sum_60'] = daily.groupby('product_id')['sold'].transform(
    lambda values: values.shift(1).rolling(60, min_periods=1).sum()
)

In [ ]:
daily['sales_std_7'] = daily.groupby('product_id')['sold'].transform(
    lambda values: values.shift(1).rolling(7, min_periods=2).std()
)
daily['max_7'] = daily.groupby('product_id')['sold'].transform(
    lambda values: values.shift(1).rolling(7, min_periods=1).max()
)

In [ ]:
daily['nonzero_7'] = daily.groupby('product_id')['sold'].transform(
    lambda values: values.shift(1).gt(0).rolling(7, min_periods=1).sum()
)
daily['nonzero_30'] = daily.groupby('product_id')['sold'].transform(
    lambda values: values.shift(1).gt(0).rolling(30, min_periods=1).sum()
)

In [ ]:
last_sale_date = daily['date'].where(daily['sold'] > 0)
last_sale_date = last_sale_date.groupby(daily['product_id']).ffill()
previous_sale_date = last_sale_date.groupby(daily['product_id']).shift(1)

daily['recency'] = (daily['date'] - previous_sale_date).dt.days.fillna(0)

In [ ]:
last_sale_qty = daily['sold'].where(daily['sold'] > 0)
last_sale_qty = last_sale_qty.groupby(daily['product_id']).ffill()
daily['last_sold'] = last_sale_qty.groupby(daily['product_id']).shift(1).fillna(0)

sales_gap = last_sale_date.groupby(daily['product_id']).diff().dt.days
sales_gap = sales_gap.where(daily['sold'] > 0)

In [ ]:
daily['lead_time'] = sales_gap.groupby(daily['product_id']).transform(
    lambda values: values.shift(1).expanding().median()
)
daily['lead_time'] = daily.groupby('product_id')['lead_time'].ffill().fillna(1)
daily['month'] = daily['date'].dt.month
daily['day_of_week'] = daily['date'].dt.dayofweek

In [ ]:
daily['future_7'] = daily.groupby('product_id')['sold'].transform(
    lambda values: values.rolling(7, min_periods=7).sum().shift(-7)
)

daily[['product_id', 'date', 'sold', 'future_7']].head(20)

In [ ]:
features = [
    'lag_1', 'lag_7', 'lag_14', 'lag_28', 'lag_56', 'lag_365',
    'mean_7', 'mean_14', 'mean_30', 'mean_60',
    'median_7', 'median_30', 'median_60',
    'sum_7', 'sum_14', 'sum_30', 'sum_60',
    'sales_std_7', 'max_7', 'nonzero_7', 'nonzero_30',
    'recency', 'last_sold', 'lead_time',
    'price', 'is_promotion', 'month', 'day_of_week'
]

daily_full = daily.copy()

In [ ]:
model_data = daily.dropna(subset=features + ['future_7']).copy()
model_data = model_data.sort_values(['date', 'product_id']).reset_index(drop=True)

print('Full daily table:', daily_full.shape)
print('Model data:', model_data.shape)

## 6. Train-Test Split

In [ ]:
all_dates = np.sort(model_data['date'].unique())
split_position = int(len(all_dates) * 0.8)
split_date = pd.Timestamp(all_dates[split_position])
train_end = split_date - pd.Timedelta(days=7)

train_mask = model_data['date'] <= train_end
test_mask = model_data['date'] > split_date

In [ ]:
x_train = model_data.loc[train_mask, features].reset_index(drop=True)
y_train = model_data.loc[train_mask, 'future_7'].reset_index(drop=True)
x_test = model_data.loc[test_mask, features].reset_index(drop=True)
y_test = model_data.loc[test_mask, 'future_7'].reset_index(drop=True)
train_dates = model_data.loc[train_mask, 'date'].reset_index(drop=True)
test_dates = model_data.loc[test_mask, 'date'].reset_index(drop=True)

print('Split date:', split_date)
print('Training through:', train_end)
print('Training rows:', x_train.shape[0])
print('Testing rows:', x_test.shape[0])

In [ ]:
def evaluate_predictions(y_true, y_pred):
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 0)
    return {
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'r2': r2_score(y_true, y_pred)
    }

## 7. Baseline and Regression Models

In [ ]:
y_pred_baseline = np.maximum(x_test['mean_30'].to_numpy() * 7, 0)
baseline_metrics = evaluate_predictions(y_test, y_pred_baseline)
baseline_metrics

In [ ]:
linear_model = LinearRegression()
linear_model.fit(x_train, y_train)

y_pred_linear = np.maximum(linear_model.predict(x_test), 0)
linear_metrics = evaluate_predictions(y_test, y_pred_linear)
linear_metrics

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=150,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(x_train, y_train)
y_pred_rf = np.maximum(rf_model.predict(x_test), 0)
rf_metrics = evaluate_predictions(y_test, y_pred_rf)
rf_metrics

## 8. XGBoost Model

In [ ]:
time_series = TimeSeriesSplit(n_splits=3, gap=7)
unique_train_dates = np.sort(train_dates.unique())
cv_splits = []

In [ ]:
for fold_train_idx, fold_valid_idx in time_series.split(unique_train_dates):
    fold_train_dates = unique_train_dates[fold_train_idx]
    fold_valid_dates = unique_train_dates[fold_valid_idx]

    fold_train_rows = np.flatnonzero(train_dates.isin(fold_train_dates).to_numpy())
    fold_valid_rows = np.flatnonzero(train_dates.isin(fold_valid_dates).to_numpy())

    cv_splits.append((fold_train_rows, fold_valid_rows))

print('Validation folds:', len(cv_splits))

In [ ]:
xgb = XGBRegressor(
    random_state=42,
    objective='reg:squarederror',
    n_jobs=1
)

param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [4, 6],
    'learning_rate': [0.05, 0.1]
}

In [ ]:
grid_xgb = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=cv_splits,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    refit=True
)

grid_xgb.fit(x_train, y_train)

In [ ]:
print('Best parameters:', grid_xgb.best_params_)
print('Best validation MAE:', -grid_xgb.best_score_)

best_xgb = grid_xgb.best_estimator_
y_pred_xgb = np.maximum(best_xgb.predict(x_test), 0)
xgb_metrics = evaluate_predictions(y_test, y_pred_xgb)
xgb_metrics

In [ ]:
model_results = pd.DataFrame([
    {'model': 'Baseline', **baseline_metrics},
    {'model': 'Linear Regression', **linear_metrics},
    {'model': 'Random Forest', **rf_metrics},
    {'model': 'XGBoost', **xgb_metrics}
])

model_results.sort_values('mae')

In [ ]:
feature_importance = pd.Series(
    best_xgb.feature_importances_,
    index=features
).sort_values(ascending=False)

feature_importance

In [ ]:
feature_importance.sort_values().plot(kind='barh', figsize=(10, 8))
plt.title('XGBoost Feature Importance')
plt.xlabel('Importance')
plt.show()

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred_xgb, alpha=0.25)
plt.xlabel('Actual 7-Day Sales')
plt.ylabel('Predicted 7-Day Sales')
plt.title('XGBoost: Actual vs Predicted')
plt.show()

## 9. Category-Level Models

In [ ]:
category_sold_train = (
    daily.loc[daily['date'] <= train_end]
         .groupby('category')['sold']
         .sum()
)

category_features = features + ['product_id', 'hcode']
category_names = sorted(model_data['category'].dropna().unique())
train_data = {}
test_data = {}

In [ ]:
for category in category_names:
    category_data = model_data[model_data['category'] == category]
    train_data[category] = category_data[category_data['date'] <= train_end].copy()
    test_data[category] = category_data[category_data['date'] > split_date].copy()

print('Categories:', len(category_names))
print('BOOKS train:', train_data.get('BOOKS', pd.DataFrame()).shape)
print('BOOKS test:', test_data.get('BOOKS', pd.DataFrame()).shape)

In [ ]:
def fit_category_model(category):
    train = train_data[category]
    test = test_data[category]
    y_test_category = test['future_7'].to_numpy()

    if train.empty:
        x_test_global = test[features].fillna(0)
        prediction = np.maximum(best_xgb.predict(x_test_global), 0)
        return None, 'global', None, prediction

    if category_sold_train.get(category, 0) < 100:
        x_test_global = test[features].fillna(0)
        prediction = np.maximum(best_xgb.predict(x_test_global), 0)
        return None, 'global', None, prediction

    x_train_category = train[category_features].copy()
    y_train_category = train['future_7'].copy()
    x_test_category = test[category_features].copy()

    x_train_category[features] = x_train_category[features].fillna(0)
    x_test_category[features] = x_test_category[features].fillna(0)

    for frame in [x_train_category, x_test_category]:
        frame['product_id'] = frame['product_id'].astype(str)
        frame['hcode'] = frame['hcode'].fillna('Unknown').astype(str)

    if y_train_category.nunique() == 1:
        constant_value = float(y_train_category.iloc[0])
        prediction = np.full(len(test), constant_value)
        return None, 'constant', constant_value, prediction

    model = CatBoostRegressor(
        iterations=150,
        depth=6,
        learning_rate=0.05,
        loss_function='RMSE',
        verbose=False,
        random_seed=42
    )

    model.fit(
        x_train_category,
        y_train_category,
        cat_features=['product_id', 'hcode']
    )

    prediction = np.maximum(model.predict(x_test_category), 0)
    return model, 'category', None, prediction

In [ ]:
category_model = {}
category_model_type = {}
category_constant = {}
category_results = []

In [ ]:
for category in category_names:
    test = test_data[category]

    if test.empty:
        continue

    model, model_type, constant_value, prediction = fit_category_model(category)
    y_test_category = test['future_7'].to_numpy()
    metrics = evaluate_predictions(y_test_category, prediction)
    if np.unique(y_test_category).size <= 1:
        metrics['r2'] = np.nan

    category_model[category] = model
    category_model_type[category] = model_type

    if model_type == 'constant':
        category_constant[category] = constant_value

    category_results.append({
        'category': category,
        'total_sold': category_sold_train.get(category, 0),
        'model_type': model_type,
        **metrics
    })

category_results = pd.DataFrame(category_results)

In [ ]:
print('Global models:', (category_results['model_type'] == 'global').sum())
print('Category models:', (category_results['model_type'] == 'category').sum())
print('Constant models:', (category_results['model_type'] == 'constant').sum())
category_results.sort_values('r2', ascending=False).head(20)

In [ ]:
print('Positive R2 categories:', (category_results['r2'] > 0).sum())
print('Negative R2 categories:', (category_results['r2'] < 0).sum())
category_results.sort_values('r2').head(20)

## 10. Final Reorder Quantity

In [ ]:
latest_date = daily_full['date'].max()
latest = daily_full[daily_full['date'] == latest_date].copy()
latest = latest.sort_values('product_id').reset_index(drop=True)

prediction_latest = np.zeros(len(latest))

In [ ]:
for category in latest['category'].dropna().unique():
    mask = latest['category'] == category
    sold_count = category_sold_train.get(category, 0)
    model_type = category_model_type.get(category, 'global')

    if sold_count < 100 or model_type == 'global':
        x_latest = latest.loc[mask, features].copy().fillna(0)
        prediction_latest[mask.to_numpy()] = np.maximum(best_xgb.predict(x_latest), 0)

    elif model_type == 'constant':
        prediction_latest[mask.to_numpy()] = category_constant[category]

    elif model_type == 'category':
        x_latest = latest.loc[mask, category_features].copy()
        x_latest[features] = x_latest[features].fillna(0)
        x_latest['product_id'] = x_latest['product_id'].astype(str)
        x_latest['hcode'] = x_latest['hcode'].fillna('Unknown').astype(str)
        prediction_latest[mask.to_numpy()] = np.maximum(
            category_model[category].predict(x_latest),
            0
        )

In [ ]:
safety_stock_latest = 0.5 * latest['sales_std_7'].fillna(0).to_numpy()

reorder_quantity_latest = np.ceil(
    np.maximum(prediction_latest + safety_stock_latest, 0)
)

In [ ]:
result_latest = latest[['category', 'product_id', 'date']].copy()
result_latest['predicted_7_day_sales'] = prediction_latest
result_latest['safety_stock'] = safety_stock_latest
result_latest['recommended_reorder_quantity'] = reorder_quantity_latest

result_latest = result_latest.sort_values(
    'recommended_reorder_quantity',
    ascending=False
).reset_index(drop=True)

result_latest.head(30)

In [ ]:
result_latest.to_csv('recommended_reorder_quantity.csv', index=False)
print('Saved recommended_reorder_quantity.csv')